# धडा 01 - AI एजंट्स चे परिचय

**AI एजंट्स फॉर बिगिनर्स** या कोर्समधील पहिल्या धड्यात तुमचे स्वागत आहे!

एक **AI एजंट** हा एक प्रोग्राम आहे जो मोठा भाषा मॉडेल (LLM) वापरतो त्याचा तर्कशक्ति इंजिन म्हणून आणि तो वास्तविक जगात *क्रिया* करू शकतो — API कॉल करणे, डेटाबेस क्वेरी करणे, किंवा कोड चालविणे — वापरकर्त्याच्या वतीने एखादे उद्दिष्ट साध्य करण्यासाठी.

या नोटबुकमध्ये तुम्ही तुमचा पहिला एजंट तयार कराल: एक **ट्रॅव्हल एजंट** जे सुट्ट्यांसाठी गंतव्य स्थळांची शिफारस करतो. याच्या दरम्यान तुम्ही शिकाल कसे:

1. **Microsoft Agent Framework** वापरून Azure AI Foundry Agent सेवा कशी कनेक्ट करायची.
2. एजंटला एक **टूल** द्यायचे — एक सोपी Python फंक्शन जी तो कॉल करू शकतो.
3. एजंट चालवायचा आणि त्याच्या उत्तराचे परीक्षण करायचे.
4. एजंटच्या उत्तराला टोकन-बाय-टोकन स्ट्रीम करायचे.


## सेटअप

हा नोटबुक चालवण्यापूर्वी, खात्री करा की आपल्याकडे:

1. **एक Azure AI Foundry प्रकल्प** ज्यामध्ये एक डिप्लॉय केलेले चॅट मॉडेल आहे (उदा. `gpt-4o-mini`).
2. **Azure CLI मध्ये लॉग इन केलेले आहे** — आपल्या टर्मिनलमध्ये `az login` चालवा.
3. **आवश्यक वातावरणीय बदल (environment variables) सेट केलेले आहेत:**
   - `AZURE_AI_PROJECT_ENDPOINT` — आपला Azure AI Foundry प्रकल्प एंडपॉइंट.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — आपल्या डिप्लॉय केलेल्या मॉडेलचे नाव.

खालील सेल आपल्याला पाहिजे असलेले Python पॅकेजेस इन्स्टॉल करतो.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## तुमचा पहिला एजंट तयार करणे

एजंटला दोन गोष्टींची गरज असते:

- **सूचना** ज्या त्याला *कोण आहे* आणि *कसे वागावे* हे सांगतात (एक सिस्टम प्रॉम्प्ट).
- **टूल्स** — Python फंक्शन्स जे `@tool` ने सजवलेले असतात आणि ज्यांना एजंट माहिती मिळवण्यासाठी किंवा क्रिया करण्यासाठी कॉल करू शकतो.

खाली आम्ही एक सोपी टूल परिभाषित करतो जी लोकप्रिय सुट्ट्यांच्या ठिकाणांची यादी परत करते. वापरकर्ता प्रवास शिफारसीसाठी विचारल्यावर एजंट हा टूल वापरेल.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## स्ट्रीमिंग प्रतिसाद

अधिक संवादात्मक अनुभवासाठी तुम्ही एजंटच्या प्रतिसादाचा **स्ट्रीम** करू शकता. पूर्ण उत्तराची वाट पाहण्याऐवजी, एजंट तयार झालेल्या मजकूराच्या तुकड्यांना थेट देतो. हे खास करून चॅट इंटरफेसमध्ये उपयुक्त आहे जिथे तुम्हाला आउटपुट वास्तविक वेळेत दाखवायचा असतो.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## सारांश

या धड्यात आपण कसे शिकले:

- **एक प्रदाता तयार करा** जो `AzureAIProjectAgentProvider` द्वारे Azure AI Foundry Agent Service शी जोडतो.
- **@tool डेकोरेटर वापरून साधन निश्चित करा** जेणेकरून एजंट आपले Python फंक्शन्स कॉल करू शकेल.
- **एजंट चालवा** वापरकर्त्याच्या संदेशासह आणि त्याचा प्रतिसाद मुद्रित करा.
- **प्रतिक्रिया स्ट्रीम करा** रिअल-टाइम आउटपुटसाठी.

पुढील धड्यात आपण एजंटिक फ्रेमवर्क्स अधिक सखोलपणे अभ्यास करू आणि एजंट्सना अधिक शक्तिशाली साधने आणि बहु-स्तरीय तर्कशक्ती क्षमता कशी द्यायची ते शिकू.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून भाषांतरित करण्यात आला आहे. आम्ही अचूकतेसाठी प्रयत्न करतो, तरीही कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये चुका किंवा अचूकतेत त्रुटी असू शकतात. मूळ दस्तऐवज त्याच्या स्थानिक भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वपूर्ण माहिती साठी व्यावसायिक मानवी भाषांतर शिफारसीय आहे. या भाषांतराच्या वापरामुळे उद्भवलेल्या कोणत्याही गैरसमजुतीसाठी किंवा चुकीच्या माहिती साठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
